# ATProto Kernel Compatibility Tests

This notebook tests the boundaries of what the WASM ObjC kernel can run from the Garazyk ATProto codebase.

## Test 1: ATURI Parsing (Pure ObjC)

In [ ]:
@interface ATURI : NSObject
@property NSString *did;
@property NSString *collection;
@property NSString *rkey;
+ (instancetype)uriWithString:(NSString *)string;
@end

@implementation ATURI
+ (instancetype)uriWithString:(NSString *)string {
    if (string == nil) { return nil; }
    if ([string hasPrefix:@"at://"] == 0) { return nil; }
    NSArray *parts = [[string substringFromIndex:5] componentsSeparatedByString:@"/"];
    if (parts.count < 3) { return nil; }
    ATURI *uri = [[ATURI alloc] init];
    uri.did = [parts[0] copy];
    uri.collection = [parts[1] copy];
    uri.rkey = [parts[2] copy];
    return uri;
}
@end

ATURI *uri = [ATURI uriWithString:@"at://did:plc:z72i7hdynmk6r5zdbiyo6mp7/app.bsky.actor.profile/self"];
NSLog(@"DID: %@", uri.did);
NSLog(@"Collection: %@", uri.collection);
NSLog(@"Rkey: %@", uri.rkey);

## Test 2: Bitwise Operators

In [ ]:
int major = 2;
int encoded = (major << 5) | 24;
NSLog(@"CBOR header: %d (0x%x)", encoded, encoded);

int byte = 0x95;
int value = byte & 0x7F;
int hasMore = byte & 0x80;
NSLog(@"Varint: %d, more: %d", value, hasMore != 0 ? 1 : 0);

int buffer = 0x48;
int shifted = buffer << 8;
NSLog(@"Shifted: %d", shifted);

int five_bits = (shifted >> 3) & 0x1F;
NSLog(@"5-bit: %d", five_bits);

NSLog(@"XOR: %d", 0x1234 ^ 0x5678);

## Test 3: NSData Byte Access

In [ ]:
NSData *data = [NSData dataWithBytes:"Hello" length:5];
NSLog(@"Length: %d", [data length]);
NSLog(@"Byte 0: %d (H)", [data byteAtIndex:0]);
NSLog(@"Byte 1: %d (e)", [data byteAtIndex:1]);

NSData *copied = [data copy];
NSLog(@"Copy equals: %d", [data isEqualToData:copied]);

NSData *data2 = [NSData dataWithBytes:" World" length:6];
NSData *combined = [data appendData:data2];
NSLog(@"Combined: %d bytes", [combined length]);

## Test 4: XRPC Dispatcher

In [ ]:
@interface XrpcDispatcher : NSObject
@property NSMutableDictionary *methodHandlers;
- (void)registerMethod:(NSString *)methodId handler:(id)handler;
- (id)handlerForMethod:(NSString *)methodId;
@end

@implementation XrpcDispatcher
- (instancetype)init {
    self = [super init];
    if (self) { _methodHandlers = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerMethod:(NSString *)methodId handler:(id)handler {
    [_methodHandlers setObject:handler forKey:methodId];
}
- (id)handlerForMethod:(NSString *)methodId {
    return [_methodHandlers objectForKey:methodId];
}
@end

XrpcDispatcher *disp = [[XrpcDispatcher alloc] init];
[disp registerMethod:@"com.atproto.server.createSession" handler:@"handler1"];
[disp registerMethod:@"com.atproto.repo.createRecord" handler:@"handler2"];

NSLog(@"Handler: %@", [disp handlerForMethod:@"com.atproto.repo.createRecord"]);

## Test 5: Category Support

In [ ]:
@interface RecordStore : NSObject
@property NSMutableDictionary *records;
- (void)putRecord:(NSString *)key value:(NSString *)value;
- (NSString *)getRecord:(NSString *)key;
@end

@implementation RecordStore
- (instancetype)init {
    self = [super init];
    if (self) { _records = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)putRecord:(NSString *)key value:(NSString *)value {
    [_records setObject:value forKey:key];
}
- (NSString *)getRecord:(NSString *)key {
    return [_records objectForKey:key];
}
@end

@interface RecordStore (Search)
- (int)recordCount;
@end

@implementation RecordStore (Search)
- (int)recordCount {
    return (int)[_records count];
}
@end

RecordStore *store = [[RecordStore alloc] init];
[store putRecord:@"app.bsky.actor.profile/self" value:@"profile_data"];
[store putRecord:@"app.bsky.feed.post/abc123" value:@"post_data"];

NSLog(@"Profile: %@", [store getRecord:@"app.bsky.actor.profile/self"]);
NSLog(@"Count: %d", [store recordCount]);

## Test 6: SHA-256 via Host Bridge

In [ ]:
NSData *input = [NSData dataWithBytes:"hello" length:5];
NSData *hash = [CID sha256Digest:input];
NSLog(@"SHA-256 hash length: %d (should be 32)", [hash length]);